# P3-06 · Train D-FINE (single class `tb_lesion`) on a T4, with auto-resume

Runs **one** config per session: `dfine_512.yaml` or `dfine_640.yaml` (identical except input size).
Train split only for training, development only for evaluation; nothing else is loaded.

**Before the first run** follow `docs/p3_kaggle_runbook.md` sections 0–4 (phone verification, rclone secret
`RCLONE_CONF`). Session options: **GPU T4**, **Internet On**.
Data: add P1's processed data (C01 split CSV, C04 PNGs + manifest, C05 parquet, C06 COCO and original full-image boxes) as a **private**
Kaggle dataset via *+ Add Input* and set `DATA_ROOT` below.

For the full run use **Save Version → Save & Run All**. If the session dies, run this notebook again
unchanged: it resumes from the last checkpoint on Drive (at most one epoch lost).

Final training is on hold pending the detector preprocessing protocol. P2 D08/D09 apply to classification.
Use the P3 training CLI's ablation options for separate synthetic or provisional detector studies.

In [ ]:
import os
CONFIG = "dfine_512.yaml"                      # or "dfine_640.yaml"
REPO = "github.com/gundatharuntej2006-hue/cv_proj.git"
BRANCH = "ritika"
DATA_ROOT = os.environ.get("TB_DATA_ROOT", "/kaggle/input/tb-processed-v1")    # TODO: folder that contains data/splits, data/processed, data/boxes
WORK = "/kaggle/working/cv_proj"

In [ ]:
import os, sys, subprocess, glob, json, hashlib, zipfile
from pathlib import Path
# Prefer the supplied local P3 snapshot so unpublished fixes are not lost.
archives = glob.glob('/kaggle/input/**/p3_code_bundle.zip', recursive=True)
if len(archives) > 1:
    raise RuntimeError('Attach exactly one P3 code bundle to avoid ambiguous versions')
if archives:
    Path(WORK).mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archives[0]) as archive:
        for entry in archive.namelist():
            candidate = (Path(WORK) / entry).resolve()
            if not candidate.is_relative_to(Path(WORK).resolve()):
                raise RuntimeError('Unsafe archive member')
        archive.extractall(WORK)
    manifest = json.loads((Path(WORK) / 'p3_bundle_manifest.json').read_text())
    for name, expected in manifest['files'].items():
        actual = hashlib.sha256((Path(WORK) / name).read_bytes()).hexdigest()
        if actual != expected:
            raise RuntimeError('P3 code bundle hash mismatch: ' + name)
    print('Code snapshot:', manifest.get('git_sha'), 'dirty:', manifest.get('working_tree_dirty'))
else:
    url = f'https://{REPO}'
    if not os.path.isdir(WORK):
        subprocess.run(['git', 'clone', '-q', '--recurse-submodules', '-b', BRANCH, url, WORK], check=True)
os.chdir(WORK)
os.environ['PYTHONPATH'] = str(Path(WORK) / 'src')
sys.path.insert(0, str(Path(WORK) / 'src'))
from p3_det.decisions import DETECTOR_PROTOCOL_PENDING
raise RuntimeError(DETECTOR_PROTOCOL_PENDING)


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements/p3.txt'], check=True)
from p3_det.setup import setup, ROOT
setup(ROOT / 'third_party/D-FINE', fetch=True)
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Enable a T4 GPU; this notebook does not substitute CPU feasibility')
print(torch.__version__, torch.version.cuda, torch.cuda.get_device_name(0))


In [ ]:
# Persistent Drive backup is required for final training in this notebook.
import shutil, urllib.request, zipfile, io
if shutil.which('rclone') is None:
    binary_dir = Path('/kaggle/working/bin')
    binary_dir.mkdir(parents=True, exist_ok=True)
    with urllib.request.urlopen('https://downloads.rclone.org/rclone-current-linux-amd64.zip') as response:
        archive_bytes = response.read()
    with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
        executable = next(name for name in archive.namelist() if name.endswith('/rclone'))
        (binary_dir / 'rclone').write_bytes(archive.read(executable))
    (binary_dir / 'rclone').chmod(0o700)
    os.environ['PATH'] = str(binary_dir) + os.pathsep + os.environ['PATH']
from p3_ops.cloud import setup_rclone_from_kaggle_secret
if not setup_rclone_from_kaggle_secret():
    raise RuntimeError('Attach the RCLONE_CONF secret before final training')
subprocess.run(['rclone', 'lsd', 'gdrive:tb_checkpoints'], check=True)


In [ ]:
os.environ['TB_DATA_ROOT'] = DATA_ROOT


In [ ]:
# Nonzero training exit stops the notebook instead of continuing to a misleading summary.
subprocess.run([sys.executable, '-m', 'p3_det.train', '--config', CONFIG], check=True)


In [ ]:
# Summary and streaming checkpoint hashes for the exact selected configuration.
import pandas as pd
from p3_det.train.config import load_run_config
from p3_det.train.run import run_dir_name
from p3_det.data.config import resolve_preprocessing
from p3_ops.cloud import CheckpointStore
cfg = load_run_config(CONFIG)
margin, clahe, provisional = resolve_preprocessing()
store = CheckpointStore.for_run(run_dir_name(cfg, margin, clahe, provisional))
rd = store.dir
print(rd)
display(pd.read_csv(rd / 'train_log.csv').tail(5))
lines = []
for checkpoint in sorted(rd.glob('ckpt_*.pt')):
    digest = hashlib.sha256()
    with checkpoint.open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    lines.append(f'{digest.hexdigest()}  {checkpoint.name}')
(rd / 'checkpoints_sha256.txt').write_text('\n'.join(lines) + '\n')
print('\n'.join(lines))
store.push()


## Bring back (after BOTH sizes finished)

From Drive `tb_checkpoints/<run folder>/` for **each** run folder (e.g. `dfine_s_512_m0.10_clahe0_PROVISIONAL`):

| File | Into the repo at |
|---|---|
| `run_meta.json`, `train_log.csv`, `dev_ground_truth.json`, `dev_detections_last.json`, `dev_detections_best.json`, `best.json`, `checkpoints_sha256.txt` | `artifacts/p3/det/runs/<run folder>/` |
| `ckpt_epochNNNN.pt`, `ckpt_best.pt` | **stay on Drive** (only their SHA-256 comes back) |

Then on the laptop: `python -m p3_det.select_size compare --run512 artifacts/p3/det/runs/<512 folder> --run640 artifacts/p3/det/runs/<640 folder>`